# 📘 Online Mean from First Principles (intuition-first)

This note is about one question:

> Why is the online (incremental) mean formula **exactly equivalent** to the normal mean?
And also:
- Why do we multiply by $1/n$?
- Why do we subtract the old mean?
- What is the *meaning* of the rearrangements (without feeling like symbol pushing)?

We’ll keep it visual and meaning-driven.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

## 1) The normal mean (batch mean)

For numbers $x_1, x_2, \dots, x_n$:

$$\bar{x}_n = \frac{x_1 + x_2 + \dots + x_n}{n}$$

It’s “sum divided by count”.

But online learning asks: if you already know $\bar{x}_{n-1}$, what happens when one new value $x_n$ arrives?

## 2) One identity that keeps the meaning alive

If the old mean is $\bar{x}_{n-1}$, then the **old sum** is:

$$S_{n-1} = (n-1)\bar{x}_{n-1}$$

This is not a trick. It is just the definition of mean rearranged:

- mean = sum / count
- so sum = mean × count

Now when a new sample $x_n$ arrives:

$$S_n = S_{n-1} + x_n$$

And the new mean is:

$$\bar{x}_n = \frac{S_n}{n} = \frac{S_{n-1} + x_n}{n}$$

## 3) The key intuition: the mean is a *weighted balance*

Substitute $S_{n-1} = (n-1)\bar{x}_{n-1}$:

$$\bar{x}_n = \frac{(n-1)\bar{x}_{n-1} + x_n}{n}$$

Now read it like a story:

- The old mean was built from **(n−1)** samples.
- The new value is **1** new sample.
- The new mean must be built from **n** samples.

So the new mean is literally a weighted average:

$$\bar{x}_n = \underbrace{\frac{n-1}{n}}_{\text{old weight}}\bar{x}_{n-1} + \underbrace{\frac{1}{n}}_{\text{new weight}}x_n$$

This directly matches your interpretation:

- Going from 5 items to 6 items: the old mean gets weight $5/6$, the new point gets weight $1/6$.
- The weights must add to 1: $\frac{n-1}{n} + \frac{1}{n} = 1$.

In [ ]:
def show_weights(n_values=(2, 3, 5, 10, 50)):
    old_w = np.array([(n-1)/n for n in n_values])
    new_w = np.array([1/n for n in n_values])

    x = np.arange(len(n_values))
    plt.figure(figsize=(9, 4))
    plt.bar(x, old_w, label='weight of old mean (n-1)/n')
    plt.bar(x, new_w, bottom=old_w, label='weight of new sample 1/n')
    plt.xticks(x, [str(n) for n in n_values])
    plt.ylim(0, 1.05)
    plt.xlabel('n (total samples after update)')
    plt.ylabel('weight')
    plt.title('Online mean is a weighted average: old gets (n−1)/n, new gets 1/n')
    plt.legend()
    plt.show()

show_weights()

## 4) Why the formula has “subtract the mean” in it

Starting from the weighted form:

$$\bar{x}_n = \frac{n-1}{n}\bar{x}_{n-1} + \frac{1}{n}x_n$$

Rewrite $\frac{n-1}{n}$ as $1 - \frac{1}{n}$ (this is not just algebra; it means **“old keeps everything except the new sample’s share”**):

$$\bar{x}_n = \left(1 - \frac{1}{n}\right)\bar{x}_{n-1} + \frac{1}{n}x_n$$

Now read it as a **replacement**:

- Remove the old mean’s share corresponding to the new sample’s weight ($\frac{1}{n}\bar{x}_{n-1}$)
- Add the new sample’s share ($\frac{1}{n}x_n$)

So the change in mean is:

$$\bar{x}_n - \bar{x}_{n-1} = \frac{1}{n}(x_n - \bar{x}_{n-1})$$

Which gives the classic online update:

$$\boxed{\bar{x}_n = \bar{x}_{n-1} + \frac{x_n - \bar{x}_{n-1}}{n}}$$

Now the subtraction has a clear meaning:

- $x_n - \bar{x}_{n-1}$ is the **gap** between the new value and where the mean currently is.
- If the new point equals the mean, the gap is 0 → the mean shouldn’t move.
- If you already have lots of data, divide by big $n$ → the mean moves only a little.

In [ ]:
def visualize_one_update(mean_old, x_new, n):
    mean_new = mean_old + (x_new - mean_old) / n

    xmin = min(mean_old, x_new, mean_new) - 2
    xmax = max(mean_old, x_new, mean_new) + 2

    plt.figure(figsize=(10, 2.2))
    plt.hlines(0, xmin, xmax, color='black', linewidth=2)
    
    plt.scatter([mean_old], [0], s=120, label='old mean')
    plt.scatter([x_new], [0], s=120, label='new sample')
    plt.scatter([mean_new], [0], s=120, label='new mean')

    plt.annotate('old mean', (mean_old, 0), xytext=(mean_old, 0.6), ha='center')
    plt.annotate('new sample', (x_new, 0), xytext=(x_new, 0.6), ha='center')
    plt.annotate('new mean', (mean_new, 0), xytext=(mean_new, -0.8), ha='center')

    plt.title(f'Update: mean_new = mean_old + (x_new - mean_old)/n  (n={n})')
    plt.yticks([])
    plt.xlim(xmin, xmax)
    plt.legend(ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.25))
    plt.show()
    return mean_new

visualize_one_update(mean_old=5.0, x_new=11.0, n=6)

## 5) Why it’s **exactly equivalent** to the normal mean

This is important: the online mean is not an approximation.
It’s the same mean, just computed in a different order.
Reason:
- We never “dropped terms”.
- We never used limits.
- We only used identities like `sum = mean × count`.
So the online update gives the exact same value as recomputing the batch mean from scratch.

In [ ]:
# Simulation: online mean equals batch mean at every step
rng = np.random.default_rng(0)
xs = rng.normal(loc=2.0, scale=3.0, size=50)

online = []
batch = []
m = 0.0
for i, x in enumerate(xs, start=1):
    if i == 1:
        m = x
    else:
        m = m + (x - m) / i
    online.append(m)
    batch.append(np.mean(xs[:i]))

online = np.array(online)
batch = np.array(batch)
max_err = np.max(np.abs(online - batch))
print(f'Max absolute difference over all steps: {max_err:.12f}')

plt.figure(figsize=(10, 5))
plt.plot(online, label='online mean', linewidth=2.5)
plt.plot(batch, '--', label='batch mean', linewidth=2.5)
plt.title('Online mean matches batch mean (step-by-step)')
plt.xlabel('step n')
plt.ylabel('mean')
plt.legend()
plt.show()

## 6) Why $1/n$ is the right amount (the cleanest intuition)
Think of the mean as a “democracy” of samples:
- After $n$ samples, each sample has weight $1/n$.
- The new sample should get exactly its share: $1/n$.
- Therefore the old information must take the remaining share: $(n-1)/n$.
That’s why $1/n$ appears. It is not magic.
It’s the sample’s fair share of influence.

## 7) Why people like online mean in practice
Even though it’s exactly the same mean, it has practical advantages:
- You don’t need to store all data.
- You don’t need huge cumulative sums (better numerical stability in many cases).
- It naturally supports streaming / real-time updates.

Online mean update:
$$\bar{x}_n \leftarrow \bar{x}_{n-1} + \frac{x_n - \bar{x}_{n-1}}{n}$$

A single line of code, but now you know what every piece means.

In [ ]:
def online_mean(values):
    m = None
    for n, x in enumerate(values, start=1):
        if n == 1:
            m = float(x)
        else:
            m = m + (x - m) / n
    return m

vals = [10, 20, 30, 40]
print('online mean:', online_mean(vals))
print('batch mean :', np.mean(vals))

# ✅ Final takeaway
The online mean is just this sentence:

> **The new mean equals the old mean plus the new sample's fair share of the gap.**
Mathematically:
$$\bar{x}_n = \bar{x}_{n-1} + \frac{1}{n}(x_n - \bar{x}_{n-1})$$
If you want next, I can write the same kind of intuition-first notebook for:
- Online variance (Welford's algorithm)
- Exponential moving average (EMA) vs true mean
- Why variance formulas can be numerically unstable